In [ ]:


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# GOLD 04 - ADOÇÃO DE TECNOLOGIAS
# ---------------------------------------------------------------------
caminho_gold_04 = PROJECT_ROOT / "Gold" / "perguntas_negocio" / "gold_04_adocao_tecnologias"

arquivos_gold_04 = [
    str(arquivo) for arquivo in caminho_gold_04.glob("part-*.csv")
]

if not arquivos_gold_04:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_04}"
    )

df_tecnologias = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_04)
)


# ---------------------------------------------------------------------
# ESTRUTURA DA GOLD
# ---------------------------------------------------------------------

"""
A inspeção inicial valida a estrutura da Gold antes das análises de adoção. O notebook confirma que a base já está agregada por edição e opção tecnológica, trazendo o número de elegíveis, quantos selecionaram a opção e o percentual de adoção calculado para cada registro.
"""
print("\n" + "=" * 100)
print("01. SCHEMA")
print("=" * 100)

df_tecnologias.printSchema()


print("\n" + "=" * 100)
print("02. COLUNAS")
print("=" * 100)

for coluna in df_tecnologias.columns:
    print(coluna)


print("\n" + "=" * 100)
print("03. AMOSTRA DOS DADOS")
print("=" * 100)

df_tecnologias.show(20, truncate=False)

"""
A amostra confirma que pct_adocao representa a relação entre selecionaram e elegiveis para cada tecnologia. Isso permite comparar a adoção proporcional entre opções e edições sem utilizar apenas as contagens absolutas de seleção.
"""


# ---------------------------------------------------------------------
# VOLUME DE DADOS
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("04. TOTAL DE LINHAS")
print("=" * 100)

print(df_tecnologias.count())

"""
A Gold 04 possui 265 registros. Como cada linha representa uma combinação analítica de edição e opção tecnológica, esse volume não deve ser interpretado como quantidade de respondentes.
"""


# ---------------------------------------------------------------------
# DISTRIBUIÇÃO POR EDIÇÃO
# ---------------------------------------------------------------------

if "edicao" in df_tecnologias.columns:
    print("\n" + "=" * 100)
    print("05. TOTAL DE LINHAS POR EDIÇÃO")
    print("=" * 100)

    (
        df_tecnologias
        .groupBy("edicao")
        .agg(F.count("*").alias("total_linhas"))
        .orderBy("edicao")
        .show(truncate=False)
    )

"""
O número de registros varia entre as edições: são 40 em 2023-2024, 115 em 2024-2025 e 110 em 2025-2026. Essa diferença indica que a quantidade de opções disponíveis na Gold não é constante no período e precisa ser considerada antes de comparações baseadas apenas em volume de linhas.
"""


# ---------------------------------------------------------------------
# NULOS POR COLUNA
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("06. NULOS POR COLUNA")
print("=" * 100)

df_tecnologias.select(
    [
        F.sum(
            F.when(F.col(coluna).isNull(), 1).otherwise(0)
        ).alias(coluna)
        for coluna in df_tecnologias.columns
    ]
).show(truncate=False)

"""
A validação não identificou valores nulos em nenhuma das seis colunas. Dessa forma, as análises seguintes não exigem tratamento de ausência para edição, opção, elegíveis, selecionados, percentual de adoção ou categoria.
"""


# ---------------------------------------------------------------------
# VALORES DISTINTOS POR COLUNA
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("07. QUANTIDADE DE VALORES DISTINTOS")
print("=" * 100)

for coluna in df_tecnologias.columns:
    total_distintos = (
        df_tecnologias
        .select(coluna)
        .distinct()
        .count()
    )

    print(f"{coluna}: {total_distintos}")

"""
A inspeção encontrou 3 edições, 92 opções tecnológicas e 7 categorias. A quantidade elevada de opções em relação ao número de categorias reforça a necessidade de analisar as tecnologias dentro de seus respectivos grupos, evitando comparar diretamente ferramentas com finalidades diferentes.
"""


# ---------------------------------------------------------------------
# CONTEÚDO DAS COLUNAS CATEGÓRICAS
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("08. VALORES DAS COLUNAS CATEGÓRICAS")
print("=" * 100)

"""
A inspeção das colunas textuais permite verificar quais tecnologias e categorias estão efetivamente disponíveis antes da construção dos rankings. A Gold reúne linguagens de programação, bancos de dados, ferramentas de BI, Cloud e dois grupos de ferramentas de ETL, além da categoria de linguagem preferida presente na base.
"""
for coluna, tipo in df_tecnologias.dtypes:
    if tipo == "string":
        print("\n" + "-" * 100)
        print(f"COLUNA: {coluna}")
        print("-" * 100)

        (
            df_tecnologias
            .groupBy(coluna)
            .agg(F.count("*").alias("contagem"))
            .orderBy(F.desc("contagem"))
            .show(100, truncate=False)
        )

"""
Os outputs também mostram que algumas opções aparecem nas três edições, como SQL, Python, Power BI e principais provedores de Cloud, enquanto outras aparecem em menos períodos. Essa diferença de disponibilidade deve ser verificada antes de utilizar uma tecnologia em análises de evolução histórica.
"""